# Spicebag: Visual Mnemonic Encoder / Decoder by Spicecips

In [ ]:
######## INSTALLERS AND LIBRARIES ########

%pip install pillow mnemonic
%pip install shamir-mnemonic
%pip install argon2-cffi
%pip install requests
%pip install ipython

from shamir_mnemonic.wordlist import WORDLIST as SLIP39_LIST
from argon2.low_level import hash_secret_raw, Type
from IPython.display import clear_output
from shamir_mnemonic import Share
from tkinter import filedialog
from mnemonic import Mnemonic
from getpass import getpass
from pathlib import Path
from PIL import Image
import tkinter as tk
import requests
import textwrap
import hashlib
import secrets
import zipfile
import bisect
import random
import struct
import hmac
import time
import io
import os


def loadElectrumWords():
    cachePath = "electrum-english.txt"

    if not os.path.exists(cachePath):
        url = "https://raw.githubusercontent.com/spesmilo/electrum/master/electrum/wordlist/english.txt"
        response = requests.get(url)

        with open(cachePath, "w", encoding = "utf-8") as f:
            f.write(response.text)

    with open(cachePath, "r", encoding = "utf-8") as f:
        return [line.strip() for line in f.readlines() if line.strip()]



######## CONSTANTS AND HELPERS ########

ALLOWED_CHUNKS = {
    b'IHDR',
    b'IDAT',
    b'IEND',
    b'tIME',
    b'tEXt',
    b'iTXt',
    b'zTXt',
    b'sRGB',
    b'gAMA',
    b'pHYs',
}

BIP39 = Mnemonic("english")

BIP39_LIST = BIP39.wordlist

ELECTRUM_LIST = loadElectrumWords()

FORBIDDEN_CHUNKS = {
    b'PLTE',
    b'tRNS',
    b'bKGD',
    b'sBIT',
    b'iCCP',
}

PERMUTATIONS = [
    (0, 1, 2),  # RGB
    (1, 2, 0),  # GBR
    (2, 0, 1),  # BRG
    (1, 0, 2),  # GRB
    (2, 1, 0),  # BGR
    (0, 2, 1),  # RBG
]

RGB_VALUE_SHIFTS = [
    (2, -7, 1), (-8, 2, -8), (1, -8, 2),
    (-8, 4, -5), (9, 0, 4), (-5, 2, -3),
    (5, -3, 6), (0, 2, -8), (7, -4, 7),
    (-1, 3, -5), (2, -6, 6), (-2, 4, -9),
    (7, -7, 5), (-7, 2, -4), (7, 0, 9),
    (-3, 6, -9), (9, -9, 5), (-9, 5, -7),
    (4, -9, 6), (-6, 9, -6), (7, -6, 2),
    (-7, 7, -2), (4, 0, 7), (-6, 6, -3),
    (0, -3, 5), (-3, 5, -4), (7, -5, 9),
    (-4, 5, -7), (1, -3, 8), (-2, 1, -7),
    (8, -5, 2), (-5, 1, -6), (6, -4, 2)
]

SEED_TYPE_STANDARDS = [
    ("BIP39", (12, 15, 18, 21, 24), BIP39_LIST, 2048, lambda s: BIP39.check(s)),
    ("ELECTRUM", (12, 24), ELECTRUM_LIST, 2048, lambda s: validateElectrum(s)),
    ("SLIP39", (20, 33), SLIP39_LIST, 1024, lambda s: all(w in SLIP39_LIST for w in s.split()))
]

WARNING_TEXT = """\
————————————————
SECURITY WARNING
————————————————————————————————————————————————————————————————————————————————————
You are about to generate or decode a visual representation of a wallet SEED PHRASE.

Proceed IF AND ONLY IF:
- You are in a private, isolated location,
- No one else can see your screen, and
- No recording / surveillance devices are present.

This program provides NO protection against malware, coercion, and espionage in any form.
—————————————————————————————————————————————————————————————————————————————————————————
"""



######## SECURITY WARNING CONFIRMATION ########

def warningConfirmation():
    clrscr()
    print(WARNING_TEXT)
    choice = input("Are you sure you want to proceed? (Y/n, case-sensitive):").strip()

    if choice != "Y":
        print("Operation aborted.")
        return False

    clrscr()
    return True



######## PNG MINIMALITY CHECKER ########

def isAcceptablePNG(path):
    with open(path, "rb") as f:
        data = f.read()

    if not data.startswith(b"\x89PNG\r\n\x1a\n"):
        return False

    i = 8
    while i + 8 <= len(data):
        length = struct.unpack(">I", data[i:i+4])[0]
        chunk = data[i+4:i+8]

        if chunk in FORBIDDEN_CHUNKS:
            return False

        i += 12 + length

    return True



######## COLOR ENCODER / DECODER ########

def computeBlockSize(maxIndex: int) -> int:
    indexBits = maxIndex.bit_length() - 1
    blockBits = 24 - indexBits
    return 1 << blockBits


def encodeWord(wordIndex, mask=0, shift=(0, 0, 0), maxIndex=2048, offset=-1):
    maskedIndex = wordIndex ^ (mask % maxIndex)

    blockSize = computeBlockSize(maxIndex)

    if offset < 0:
        offset = secrets.randbelow(blockSize)

    base = maskedIndex * blockSize + offset

    r = (base >> 16) & 0xFF
    g = (base >> 8) & 0xFF
    b = base & 0xFF

    r = (r + shift[0]) % 256
    g = (g + shift[1]) % 256
    b = (b + shift[2]) % 256

    permIndex = (r + g + b) % 6
    perm = PERMUTATIONS[permIndex]

    rgb = [r, g, b]
    return tuple(rgb[p] for p in perm)


def decodeColor(rgb, mask=0, shift=(0, 0, 0), maxIndex=2048):
    r, g, b = rgb

    permIndex = (r + g + b) % 6
    perm = PERMUTATIONS[permIndex]

    canonical = [0, 0, 0]
    for i, p in enumerate(perm):
        canonical[p] = rgb[i]

    R, G, B = canonical

    R = (R - shift[0]) % 256
    G = (G - shift[1]) % 256
    B = (B - shift[2]) % 256

    key = (R << 16) | (G << 8) | B
    blockSize = computeBlockSize(maxIndex)

    maskedIndex = key // blockSize
    wordIndex = maskedIndex ^ (mask % maxIndex)

    if wordIndex >= maxIndex:
        raise ValueError("PNG contains forbidden chunks or unacceptable color space.")

    return wordIndex



######## SEED ENCODER ########

def deriveMasterKey(salt: str) -> bytes:
    if not salt:
        return b"\x00" * 32

    saltBytes = salt.encode("utf-8")

    return hash_secret_raw(
        secret=saltBytes,
        salt=b"SpicebagDomain",
        time_cost=4,
        memory_cost=2**18,
        parallelism=1,
        hash_len=64,
        type=Type.ID
    )


def hkdfExpand(key: bytes, info: bytes, length: int = 128) -> bytes:
    prk = hmac.new(b"\x00" * 64, key, hashlib.sha512).digest()

    t = b""
    okm = b""
    counter = 1

    while len(okm) < length:
        t = hmac.new(prk, t + info + bytes([counter]), hashlib.sha512).digest()
        okm += t
        counter += 1

    return okm[:length]


def deriveSubkeys(masterKey: bytes):
    maskKey = hkdfExpand(masterKey, b"maskDomain")
    permKey = hkdfExpand(masterKey, b"permDomain")
    offsetKey = hkdfExpand(masterKey, b"offsetDomain")

    return maskKey, permKey, offsetKey


def deriveMask(maskKey: bytes, index: int, maxIndex: int) -> int:
    msg = index.to_bytes(4, "big")
    digest = hmac.new(maskKey, msg, hashlib.sha512).digest()
    return int.from_bytes(digest[:4], "big") % maxIndex



######## PNG GENERATOR ########

def encodeMnemonic(mnemonicRaw, imgPath, cellPx=100, salt="", precomputed=None, wordOffsets=None):
    indices, standard, numWords, maxIdx = identifySeedType(mnemonicRaw)

    if standard == "SLIP39":
        if numWords == 20:
            cols, rows = (4, 5)

        else:
            cols, rows = (3, 11)

    else:
        mapping = {
            12: (3, 4),
            15: (3, 5),
            18: (3, 6),
            21: (3, 7),
            24: (4, 6)
        }

        cols, rows = mapping[numWords]

    if precomputed is not None:
        maskKey, rngSeed = precomputed

    elif salt:
        masterKey = deriveMasterKey(salt)
        maskKey, permKey, _ = deriveSubkeys(masterKey)
        rngSeed = int.from_bytes(permKey[:8], "big")

    else:
        maskKey = None
        rngSeed = None

    if wordOffsets is None:
        blockSize = computeBlockSize(maxIdx)
        rng = random.Random(rngSeed) if rngSeed is not None else secrets.SystemRandom()
        wordOffsets = [rng.randrange(blockSize) for _ in range(numWords)]

    allCoords = [(r, c) for r in range(rows) for c in range(cols)]

    if rngSeed is not None:
        random.Random(rngSeed).shuffle(allCoords)

    img = Image.new("RGB", (cols * cellPx, rows * cellPx))
    px = img.load()
    assert px is not None

    for i in range(numWords):
        wordMask = deriveMask(maskKey, i, maxIdx) if maskKey is not None else 0
        currentShift = RGB_VALUE_SHIFTS[i]

        color = encodeWord(
            indices[i],
            mask=wordMask,
            shift=currentShift,
            maxIndex=maxIdx,
            offset=wordOffsets[i]
        )

        r, c = allCoords[i]

        for y in range(r * cellPx, (r + 1) * cellPx):
            for x in range(c * cellPx, (c + 1) * cellPx):
                px[x, y] = color

    img.save(imgPath, format="PNG", optimize=False)


def bulkEncodeMnemonic(mnemonicRaw, zipPath, count, cellPx=100, salt=""):
    indices, standard, numWords, maxIdx = identifySeedType(mnemonicRaw)

    if salt:
        masterKey = deriveMasterKey(salt)
        maskKey, permKey, _ = deriveSubkeys(masterKey)
        rngSeed = int.from_bytes(permKey[:8], "big")
        precomputed = (maskKey, rngSeed)
        rng = random.Random(rngSeed)

    else:
        precomputed = None
        rng = secrets.SystemRandom()

    blockSize = computeBlockSize(maxIdx)
    allWordOffsets = []

    for _ in range(numWords):
        if count <= blockSize:
            allWordOffsets.append(rng.sample(range(blockSize), count))

        else:
            allWordOffsets.append([rng.randrange(blockSize) for _ in range(count)])

    with zipfile.ZipFile(zipPath, 'w', compression=zipfile.ZIP_STORED) as zipf:
        for i in range(count):
            imgOffsets = [allWordOffsets[w][i] for w in range(numWords)]
            imgByteArr = io.BytesIO()
            encodeMnemonic(mnemonicRaw, imgByteArr, cellPx, salt, precomputed, imgOffsets)

            imgByteArr.seek(0)
            zipf.writestr(f"{i + 1}.png", imgByteArr.getvalue())
            imgByteArr.close()



######## IMAGE DECODER ########

def extractRGB(pixel):
    if not isinstance(pixel, tuple):
        raise ValueError("Invalid pixel format.")

    if len(pixel) == 3:
        r, g, b = pixel

    elif len(pixel) == 4:
        r, g, b, a = pixel

        if a != 255:
            raise ValueError("Alpha channel must be fully opaque (0xFF).")

    else:
        raise ValueError("Unsupported color model.")

    for v in (r, g, b):
        if type(v) is not int or not (0 <= v <= 255):
            raise ValueError("Color channel is outside semantic 8-bit range.")

    return (r, g, b)


def decodeImage(imagePath, salt=""):
    img = Image.open(imagePath)
    pixels = img.load()
    assert pixels is not None

    width, height = img.size

    cols, rows, numWords, maxIdx = getGridDimensions(width, height)

    cellWidth = width // cols
    cellHeight = height // rows

    if salt:
        masterKey = deriveMasterKey(salt)
        maskKey, permKey, _ = deriveSubkeys(masterKey)
        rngSeed = int.from_bytes(permKey[:8], "big")

    else:
        maskKey = None
        rngSeed = None

    allCoords = [(r, c) for r in range(rows) for c in range(cols)]

    if rngSeed is not None:
        random.Random(rngSeed).shuffle(allCoords)

    wordIndices = []

    for i in range(numWords):
        r, c = allCoords[i]

        basePx = pixels[c * cellWidth, r * cellHeight]
        baseColor = extractRGB(basePx)

        wordMask = deriveMask(maskKey, i, maxIdx) if maskKey is not None else 0
        currentShift = RGB_VALUE_SHIFTS[i]

        wordIndices.append(
            decodeColor(
                baseColor,
                mask=wordMask,
                shift=currentShift,
                maxIndex=maxIdx
            )
        )

    if numWords in (12, 15, 18, 21, 24):
        mnemonic = " ".join(BIP39_LIST[idx] for idx in wordIndices)

        if validateBIP39(mnemonic):
            return mnemonic

    if numWords in (12, 24):
        mnemonic = " ".join(ELECTRUM_LIST[idx] for idx in wordIndices)

        if validateElectrum(mnemonic):
            return mnemonic

    if numWords in (20, 33):
        mnemonic = " ".join(SLIP39_LIST[idx] for idx in wordIndices)

        try:
            Share.from_mnemonic(mnemonic)
            return mnemonic

        except Exception:
            pass

    raise ValueError(
        "Mnemonic checksum failed.\n"
        "—————> Image or salt is incorrect."
    )



######## SEED PHRASE .TXT FILE FORMATTER ########

def formatSeedFile(mnemonic: str) -> str:
    return mnemonic.strip()


def writeSeedFile(path: str, mnemonic: str):
    with open(path, "w", encoding="utf-8") as f:
        f.write(formatSeedFile(mnemonic))



######## CLI ENTRY POINT ########

def promptChoice():
    print("Spicebag by Spicecips")
    print("—————————————————————")
    print("1) Generate image from seed phrase")
    print("2) Generate image from seed phrase (bulk)")
    print("3) Decode seed phrase from image")
    print("0) Exit")
    print("———————")

    choice = input("\nSelect option 1/2/3/0:").strip()
    return choice



######## MNEMONIC VALIDATOR ########

def normalizeMnemonicInput(raw: str) -> list[str]:
    return raw.lower().split()


def validateBIP39(mnemonicStr: str) -> bool:
    return BIP39.check(mnemonicStr)


def validateElectrum(mnemonic: str) -> bool:
    try:
        h = hmac.new(b"Seed version", mnemonic.encode('utf-8'), hashlib.sha512).hexdigest()
        return h.startswith(('01', '100', '101'))

    except:
        return False


def binarySearchWord(wordList: list, word: str) -> int:
    idx = bisect.bisect_left(wordList, word)

    if idx < len(wordList) and wordList[idx] == word:
        return idx

    return -1


def identifySeedType(rawInput: str):
    words = rawInput.lower().split()
    mnemonicStr = " ".join(words)
    numWords = len(words)

    candidateStandards = [s for s in SEED_TYPE_STANDARDS if numWords in s[1]]

    if not candidateStandards:
        raise ValueError(
            f"Invalid mnemonic length: {numWords} word(s) inputted.\n"
            "—————> Valid lengths are 12, 15, 18, 20, 21, 24, or 33 words."
        )

    allValidWords = set()

    for _, _, wordList, _, _ in candidateStandards:
        allValidWords.update(wordList)

    invalidWords = []
    seen = set()

    for w in words:
        if w not in allValidWords and w not in seen:
            invalidWords.append(w)
            seen.add(w)

    if invalidWords:
        invalidWords.sort()
        maskedWords = [f"{w[0]}•••{w[-1]}" for w in invalidWords]

        raise ValueError(
            f"Invalid seed word(s) or character(s).\n"
            "—————> "
            + ", ".join(maskedWords)
            + "."
        )

    for name, _, wordList, maxIdx, validator in candidateStandards:
        if all(w in wordList for w in words):
            if name == "SLIP39":
                try:
                    Share.from_mnemonic(mnemonicStr)
                    indices = [binarySearchWord(wordList, w) for w in words]

                    return indices, name, numWords, maxIdx

                except Exception:
                    pass

            elif validator(mnemonicStr):
                indices = [binarySearchWord(wordList, w) for w in words]

                return indices, name, numWords, maxIdx

    raise ValueError(
        "Mnemonic checksum failed.\n"
        "—————> Unrecognized or invalid seed phrase."
    )


def getGridDimensions(width, height):
    if width * 4 == height * 3:
        return 3, 4, 12, 2048

    if width * 5 == height * 3:
        return 3, 5, 15, 2048

    if width * 2 == height:
        return 3, 6, 18, 2048

    if width * 5 == height * 4:
        return 4, 5, 20, 1024

    if width * 7 == height * 3:
        return 3, 7, 21, 2048

    if width * 3 == height * 2:
        return 4, 6, 24, 2048

    if width * 11 == height * 3:
        return 3, 11, 33, 1024

    raise ValueError("Invalid image aspect ratio.")


def validateImage(path):
    if not isAcceptablePNG(path):
        raise ValueError("PNG contains forbidden chunks or unacceptable color space.")

    img = Image.open(path)
    pixels = img.load()
    assert pixels is not None

    width, height = img.size

    cols, rows, numWords, _ = getGridDimensions(width, height)

    if width % cols != 0 or height % rows != 0:
        raise ValueError("Invalid cell aspect ratio.")

    cellWidth = width // cols
    cellHeight = height // rows

    for r in range(rows):
        for c in range(cols):

            basePx = pixels[c * cellWidth, r * cellHeight]
            baseColor = extractRGB(basePx)

            for y in range(r * cellHeight, (r + 1) * cellHeight):
                for x in range(c * cellWidth, (c + 1) * cellWidth):
                    if extractRGB(pixels[x, y]) != baseColor:
                        raise ValueError(
                            "Non-monochromatic cell(s) detected.\n"
                            "—————> Image might have been compressed."
                        )

    return True



######## INPUT HELPERS ########

def clrscr():
    clear_output(wait=False)


def promptCellSize(): # Option 1 / 2
    val = input("Cell size in pixels:").strip()

    if not val.isdigit() or int(val) < 1:
        raise ValueError("Invalid cell size.")

    return int(val)


def pickPNGSavePath(defaultFilename=""): # Option 1
    print("Select PNG save path...")
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    filePath = filedialog.asksaveasfilename(
        title="Save PNG image",
        initialfile=defaultFilename,
        defaultextension=".png",
        filetypes=[("PNG images", "*.png")],
    )

    root.destroy()
    return filePath or None


def pickBulkPNGSavePath(defaultFilename=""):
    print("Select ZIP save path...")
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    zipPath = filedialog.asksaveasfilename(
        title="Save bulk images ZIP",
        initialfile=defaultFilename,
        defaultextension=".zip",
        filetypes=[("ZIP files", "*.zip")],
    )

    root.destroy()
    return zipPath or None


def pickPNGFile(): # Option 3
    print("\nSelect color-coded PNG...")
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    filePath = filedialog.askopenfilename(
        title="Select PNG image",
        filetypes=[("PNG images", "*.png")],
    )

    root.destroy()
    return filePath or None


def promptSaveSeedToFile() -> bool: # Option 3
    choice = input("Save seed phrase to a .txt file? (Y/n, case-sensitive):").strip()

    if choice != "Y":
        clrscr()
        print("Operation aborted.\n")
        return False

    return True


def pickSeedSavePath(): # Option 3
    print("Select TXT save path...")
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    filePath = filedialog.asksaveasfilename(
        title="Save TXT file",
        defaultextension=".txt",
        filetypes=[("Text files", "*.txt")],
    )

    root.destroy()
    return filePath or None



######## MAIN CONTROL FLOW ########

def main():
    if not warningConfirmation():
        return

    while True:
        try:
            choice = promptChoice()

            if choice == "1":
                mnemonicRaw = input("\nEnter seed phrase:").strip()
                identifySeedType(mnemonicRaw)

                salt = getpass("Salt (optional extra security, leave blank for none):")

                cellPx = promptCellSize()
                wordCount = len(mnemonicRaw.split())
                timestamp = time.strftime("%Y%m%d_%H%M%S")
                defaultFilename = f"SeedImage{wordCount}_{timestamp}.png"
                imgPath = pickPNGSavePath(defaultFilename)

                if imgPath is None:
                    clrscr()
                    print("Save canceled.\n")
                    continue

                encodeMnemonic(mnemonicRaw, imgPath, cellPx, salt)
                clrscr()

                if salt:
                    successSaltMsg=textwrap.dedent(f"""\
                    {Path(imgPath).as_uri()} is generated successfully and encrypted with your salt.

                    IMPORTANT: Memorize or store your salt safely.
                    Loss of salt will result in image decoding failure.
                    """)
                    print(successSaltMsg)

                else:
                    print(f"{Path(imgPath).as_uri()} is generated successfully!\n")

            elif choice == "2":
                try:
                    count = int(input("\nHow many images do you want to generate?:").strip())

                    if count < 1:
                        raise ValueError

                except ValueError:
                    raise ValueError("Invalid amount. Please enter a positive integer.")

                mnemonicRaw = input("Enter seed phrase:").strip()
                identifySeedType(mnemonicRaw)

                salt = getpass("Salt (optional extra security, leave blank for none):")

                cellPx = promptCellSize()
                wordCount = len(mnemonicRaw.split())
                timestamp = time.strftime("%Y%m%d_%H%M%S")
                defaultFilename = f"SeedImages{wordCount}x{count}_{timestamp}.zip"
                zipPath = pickBulkPNGSavePath(defaultFilename)

                if zipPath is None:
                    clrscr()
                    print("Save canceled.\n")
                    continue

                bulkEncodeMnemonic(mnemonicRaw, zipPath, count, cellPx, salt)
                clrscr()

                if salt:
                    successSaltMsg=textwrap.dedent(f"""\
                    {count} images are packaged successfully into {Path(zipPath).as_uri()} and encrypted with your salt.

                    IMPORTANT: Memorize or store your salt safely.
                    Loss of salt will result in image decoding failure.
                    """)
                    print(successSaltMsg)

                else:
                    print(f"{count} images are packaged successfully into {Path(zipPath).as_uri()}!\n")

            elif choice == "3":
                path = pickPNGFile()

                if path is None:
                    clrscr()
                    print("No file selected.\n")
                    continue

                validateImage(path)

                salt = getpass("Salt (leave blank if none was used): ")
                mnemonic = decodeImage(path, salt)

                if promptSaveSeedToFile():
                    seedPath = pickSeedSavePath()

                    if seedPath is None:
                        clrscr()
                        print("Save canceled.\n")
                        continue

                    writeSeedFile(seedPath, mnemonic)
                    clrscr()
                    print(f"Seed phrase is successfully written to {Path(seedPath).as_uri()}!\n")

            elif choice == "0":
                print("Program exited successfully.")
                return

            else:
                raise ValueError(
                    "Empty or invalid selection.\n"
                    "—————> Choose between 1, 2, 3, and 0."
                )

        except Exception as e:
            clrscr()
            print("ERROR:", e, "\n")
            continue



######## BOILERPLATE RUNNER ########

if __name__ == "__main__":
    main()